Why a Checkpointer (MemorySaver) is mandatory: LangGraph needs to save the current State when it pauses so it remembers everything when the human replies later.

Two Ways Krish Shows HITL:

Method A (interrupt_before=["tools"]): You pass interrupt_before=["tools"] inside builder.compile(...). Whenever the LLM decides to call a tool, LangGraph automatically freezes right before running the tools node. To resume, you simply call graph.invoke(None, config) (passing None as input means "continue from where you paused").

Method B (interrupt(...) + Command(resume=...)): Inside a custom node (like human_assistance), you call interrupt({"query": query}). LangGraph pauses and returns the question to the user; you resume by passing Command(resume={"data": "your human answer"}).

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

load_dotenv(override=True)

True

In [2]:
@tool
def human_assistance(query: str) -> str:
    """Request assistance from a human."""
    human_response = interrupt({"query": query})
    return human_response["data"]

tavily_tool = TavilySearch(max_results=2)
tools = [tavily_tool, human_assistance]

llm = init_chat_model("google_genai:gemini-3.5-flash-lite")
llm_with_tools = llm.bind_tools(tools)

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    # Disable parallel tool calling when using human interrupts so one interrupt runs cleanly at a time
    return {"messages": [message]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [4]:
config = {"configurable": {"thread_id": "1"}}

user_input = "I need some expert guidance for building an AI agent. Could you request assistance for me?"

events = graph.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    config,
    stream_mode="values",
)
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need some expert guidance for building an AI agent. Could you request assistance for me?
================================== Ai Message ==================================

[]
Tool Calls:
  human_assistance (call_1768746475364854004)
 Call ID: call_1768746475364854004
  Args:
    query: User needs expert guidance for building an AI agent.
================================== Ai Message ==================================

[]
Tool Calls:
  human_assistance (call_1768746475364854004)
 Call ID: call_1768746475364854004
  Args:
    query: User needs expert guidance for building an AI agent.


In [5]:
snapshot = graph.get_state(config)
print("Next node waiting to run:", snapshot.next)

Next node waiting to run: ('tools',)


In [6]:
human_response = (
    "We, the experts, are here to help! We'd recommend you check out LangGraph to build your agent. "
    "It's much more reliable and extensible than simple autonomous agents."
)

human_command = Command(resume={"data": human_response})

events = graph.stream(human_command, config, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  human_assistance (call_1768746475364854004)
 Call ID: call_1768746475364854004
  Args:
    query: User needs expert guidance for building an AI agent.
================================= Tool Message =================================
Name: human_assistance

We, the experts, are here to help! We'd recommend you check out LangGraph to build your agent. It's much more reliable and extensible than simple autonomous agents.
================================== Ai Message ==================================

[{'type': 'text', 'text': "Our resident experts are here to help! They recommend checking out **LangGraph** to build your agent, as it provides much more reliability and extensibility than standard autonomous agent frameworks. \n\nTo help us tailor the guidance specifically to your project, could you tell us a bit more about what you're trying to build? For example:\n1. **What is the core purpos